## 🔧 Cell 1: Check GPU

In [ ]:
import subprocess
import torch

print("="*60)
print("   SYSTEM INFO")
print("="*60)

# GPU Check
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"\n✅ GPU Available: {gpu_name}")
    print(f"   Memory: {gpu_memory:.1f} GB")
else:
    print("\n❌ GPU NOT DETECTED!")
    print("→ Go to Settings → Accelerator → Select GPU")

# Disk space
import shutil
total, used, free = shutil.disk_usage("/")
print(f"\n📁 Disk Space: {free // (2**30)} GB free")

## 📦 Cell 2: Install Dependencies
This will take ~3-5 minutes

In [ ]:
%%time
print("="*60)
print("   INSTALLING DEPENDENCIES")
print("="*60)

# Core dependencies
!pip install -q diffusers==0.25.0
!pip install -q transformers==4.36.0
!pip install -q accelerate
!pip install -q safetensors
!pip install -q Pillow
!pip install -q scipy

print("\n✅ All dependencies installed!")

## 🔍 Cell 3: Find Your Uploaded Images

In [ ]:
import os
import glob
from PIL import Image
from IPython.display import display

print("="*60)
print("   FINDING YOUR FILES")
print("="*60)

INPUT_DIR = "/kaggle/input"
WORKING_DIR = "/kaggle/working"

# Find images
images = glob.glob(f"{INPUT_DIR}/**/*.jpg", recursive=True) + \
         glob.glob(f"{INPUT_DIR}/**/*.jpeg", recursive=True) + \
         glob.glob(f"{INPUT_DIR}/**/*.png", recursive=True) + \
         glob.glob(f"{INPUT_DIR}/**/*.webp", recursive=True)

print(f"\n📸 Found {len(images)} image(s):")
for img_path in images:
    size_kb = os.path.getsize(img_path) / 1024
    print(f"   • {img_path} ({size_kb:.1f} KB)")

# Set default
INPUT_IMAGE = images[0] if images else None

if INPUT_IMAGE:
    print(f"\n✅ Using: {INPUT_IMAGE}")
    # Show preview
    img = Image.open(INPUT_IMAGE)
    img.thumbnail((400, 400))
    display(img)
else:
    print("\n❌ No images found! Please upload an image.")

## ⚙️ Cell 4: Configuration

### 🎯 Available Models

| Model | Speed | Quality | Best For |
|-------|-------|---------|----------|
| `instruct-pix2pix` | Fast (~5-10s) | Good | General edits |
| `instruct-pix2pix-xl` | Slow (~30-60s) | Best | High quality edits |

### 💡 Example Instructions:
- Style: "Make it a watercolor painting", "Turn into anime style"
- Weather: "Add rain", "Make it snowy", "Add fog"
- Time: "Make it night time", "Make it sunset"
- Transform: "Turn him into a cyborg", "Make her look older"
- Add/Remove: "Add sunglasses", "Remove the background"

In [ ]:
#============================================================
#   CONFIGURATION
#============================================================

# Input image path (auto-detected above, or set manually)
# INPUT_IMAGE = "/kaggle/input/your-dataset/image.jpg"

# Output path
OUTPUT_IMAGE = "/kaggle/working/edited_image.png"

#============================================================
#   ✏️ YOUR EDIT INSTRUCTION (Modify this!)
#============================================================

INSTRUCTION = "Make it a sunset scene"

# More examples:
# INSTRUCTION = "Turn it into a watercolor painting"
# INSTRUCTION = "Add snow everywhere"
# INSTRUCTION = "Make the person look like a zombie"
# INSTRUCTION = "Replace the sky with stars"
# INSTRUCTION = "Make it look like a vintage photo"
# INSTRUCTION = "Add dramatic lighting"

#============================================================
#   MODEL SETTINGS
#============================================================

# Model choice
MODEL = "instruct-pix2pix"  # Fast, good quality
# MODEL = "instruct-pix2pix-xl"  # Slower, better quality

# Generation parameters
IMAGE_GUIDANCE_SCALE = 1.5   # How much to preserve original (1.0-2.0)
GUIDANCE_SCALE = 7.5         # How closely to follow instruction (5-15)
NUM_INFERENCE_STEPS = 20     # Quality steps (10-50, more = better but slower)
NUM_IMAGES = 1               # Number of variations to generate (1-4)

#============================================================

print("="*60)
print("   📋 CONFIGURATION")
print("="*60)
print(f"\n   📸 Input:       {INPUT_IMAGE}")
print(f"   💾 Output:      {OUTPUT_IMAGE}")
print(f"\n   ✏️  Instruction: '{INSTRUCTION}'")
print(f"\n   🤖 Model:       {MODEL}")
print(f"   ⚙️  Image Guidance: {IMAGE_GUIDANCE_SCALE}")
print(f"   ⚙️  Text Guidance:  {GUIDANCE_SCALE}")
print(f"   ⚙️  Steps:         {NUM_INFERENCE_STEPS}")
print("="*60)

## 🎨 Cell 5: Load Model

In [ ]:
%%time
import torch
from diffusers import StableDiffusionInstructPix2PixPipeline, EulerAncestralDiscreteScheduler
from PIL import Image

print("="*60)
print("   LOADING MODEL")
print("="*60)

# Model mapping
model_ids = {
    "instruct-pix2pix": "timbrooks/instruct-pix2pix",
    "instruct-pix2pix-xl": "diffusers/sdxl-instructpix2pix-768"
}

model_id = model_ids.get(MODEL, model_ids["instruct-pix2pix"])
print(f"\n   Loading: {model_id}")

# Load pipeline
pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None
)
pipe.to("cuda")
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)

# Enable memory optimizations
pipe.enable_attention_slicing()
try:
    pipe.enable_xformers_memory_efficient_attention()
    print("   ✅ xformers enabled")
except:
    print("   ⚠️ xformers not available")

print("\n✅ Model loaded!")

## 🚀 Cell 6: Edit Image!

In [ ]:
%%time
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt

print("="*60)
print("   ✨ EDITING IMAGE")
print("="*60)
print(f"\n   Instruction: '{INSTRUCTION}'")

# Load input image
input_image = Image.open(INPUT_IMAGE).convert("RGB")

# Resize if too large (to save memory)
max_size = 768
if max(input_image.size) > max_size:
    ratio = max_size / max(input_image.size)
    new_size = (int(input_image.width * ratio), int(input_image.height * ratio))
    # Make dimensions divisible by 8
    new_size = (new_size[0] - new_size[0] % 8, new_size[1] - new_size[1] % 8)
    input_image = input_image.resize(new_size, Image.LANCZOS)
    print(f"   Resized to: {new_size}")

# Generate
print(f"\n   Generating {NUM_IMAGES} image(s)...")

results = pipe(
    INSTRUCTION,
    image=input_image,
    num_inference_steps=NUM_INFERENCE_STEPS,
    image_guidance_scale=IMAGE_GUIDANCE_SCALE,
    guidance_scale=GUIDANCE_SCALE,
    num_images_per_prompt=NUM_IMAGES
).images

# Save results
print("\n   Saving results...")
for i, result in enumerate(results):
    if i == 0:
        output_path = OUTPUT_IMAGE
    else:
        output_path = OUTPUT_IMAGE.replace(".png", f"_{i+1}.png")
    result.save(output_path)
    print(f"   ✅ Saved: {output_path}")

# Display comparison
print("\n" + "="*60)
print("   📊 BEFORE vs AFTER")
print("="*60)

fig, axes = plt.subplots(1, 1 + len(results), figsize=(6 * (1 + len(results)), 6))
if len(results) == 1:
    axes = [axes] if not isinstance(axes, list) else axes

# Original
axes[0].imshow(input_image)
axes[0].set_title("Original", fontsize=14)
axes[0].axis("off")

# Results
for i, result in enumerate(results):
    axes[i+1].imshow(result)
    axes[i+1].set_title(f"Edited: '{INSTRUCTION}'", fontsize=12)
    axes[i+1].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "🎉"*20)
print("   EDITING COMPLETE!")
print("🎉"*20)

## 🔄 Cell 7: Try Different Instructions (Optional)
Run this cell multiple times with different instructions!

In [ ]:
#============================================================
#   TRY DIFFERENT INSTRUCTIONS
#============================================================

# Change this and run the cell again!
NEW_INSTRUCTION = "Turn it into a pencil sketch"

# Adjust these for different results
NEW_IMAGE_GUIDANCE = 1.5  # 1.0 = more creative, 2.0 = preserve more
NEW_TEXT_GUIDANCE = 7.5   # Higher = follow instruction more strictly

#============================================================

print(f"Generating: '{NEW_INSTRUCTION}'...")

result = pipe(
    NEW_INSTRUCTION,
    image=input_image,
    num_inference_steps=NUM_INFERENCE_STEPS,
    image_guidance_scale=NEW_IMAGE_GUIDANCE,
    guidance_scale=NEW_TEXT_GUIDANCE,
).images[0]

# Save with instruction name
safe_name = NEW_INSTRUCTION.replace(" ", "_")[:30]
output_path = f"/kaggle/working/edited_{safe_name}.png"
result.save(output_path)
print(f"✅ Saved: {output_path}")

# Display
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(input_image)
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(result)
axes[1].set_title(f"'{NEW_INSTRUCTION}'")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 🎨 Cell 8: Batch Edit with Multiple Instructions

In [ ]:
#============================================================
#   BATCH EDIT - Apply multiple instructions at once
#============================================================

BATCH_INSTRUCTIONS = [
    "Make it a watercolor painting",
    "Add dramatic sunset lighting",
    "Turn it into anime style",
    "Make it look vintage",
]

#============================================================

print("="*60)
print("   🎨 BATCH EDITING")
print("="*60)

batch_results = []

for i, instruction in enumerate(BATCH_INSTRUCTIONS):
    print(f"\n[{i+1}/{len(BATCH_INSTRUCTIONS)}] '{instruction}'...")
    
    result = pipe(
        instruction,
        image=input_image,
        num_inference_steps=15,  # Fewer steps for faster batch
        image_guidance_scale=1.5,
        guidance_scale=7.5,
    ).images[0]
    
    batch_results.append((instruction, result))
    
    # Save
    safe_name = instruction.replace(" ", "_")[:25]
    result.save(f"/kaggle/working/batch_{i+1}_{safe_name}.png")

# Display all results
print("\n" + "="*60)
print("   📊 ALL RESULTS")
print("="*60)

n_cols = min(len(BATCH_INSTRUCTIONS) + 1, 3)
n_rows = (len(BATCH_INSTRUCTIONS) + 1 + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5*n_rows))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

# Original
axes[0].imshow(input_image)
axes[0].set_title("Original", fontsize=12)
axes[0].axis("off")

# Results
for i, (instruction, result) in enumerate(batch_results):
    axes[i+1].imshow(result)
    axes[i+1].set_title(instruction[:30], fontsize=10)
    axes[i+1].axis("off")

# Hide empty axes
for i in range(len(batch_results) + 1, len(axes)):
    axes[i].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/batch_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Batch editing complete!")

## 📥 Cell 9: Download Results

In [ ]:
import os

print("📁 Output files:")
print("-" * 40)

for f in sorted(os.listdir(WORKING_DIR)):
    path = os.path.join(WORKING_DIR, f)
    if os.path.isfile(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"   • {f} ({size_kb:.1f} KB)")

print("\n💡 To download: Click 'Output' tab on the right → Click on the file")

---

## 📚 Tips & Tricks

### 🎯 Writing Better Instructions

**Be specific:**
- ❌ "Make it better"
- ✅ "Add warm golden sunset lighting"

**Describe the change:**
- ❌ "Change it"
- ✅ "Turn the day scene into night with stars"

### ⚙️ Parameter Tuning

| Parameter | Low Value | High Value |
|-----------|-----------|------------|
| `image_guidance_scale` | More creative changes | Preserve original more |
| `guidance_scale` | Subtle edits | Strong edits |
| `num_inference_steps` | Faster, lower quality | Slower, higher quality |

### 💡 Common Instructions That Work Well

**Style Transfer:**
- "Make it a watercolor painting"
- "Turn into oil painting style"
- "Convert to anime/manga style"
- "Make it look like a pencil sketch"
- "Apply pop art style"

**Lighting/Weather:**
- "Make it sunset/sunrise"
- "Add dramatic lighting"
- "Make it rainy/snowy"
- "Add fog/mist"
- "Make it night time"

**Transformations:**
- "Make the person older/younger"
- "Add a smile"
- "Turn into a zombie/robot"
- "Add sunglasses/hat"

**Color/Mood:**
- "Make it black and white"
- "Add vintage/retro look"
- "Make colors more vibrant"
- "Add sepia tone"

---

## ❓ Troubleshooting

### "Out of memory"
- Reduce image size (max 512x512)
- Reduce `num_inference_steps` to 10-15
- Restart kernel

### "Results don't match instruction"
- Increase `guidance_scale` (try 10-15)
- Be more specific in instruction
- Try different wording

### "Too different from original"
- Increase `image_guidance_scale` (try 1.8-2.0)
- Reduce `guidance_scale`

---

**Happy Editing! ✨**